In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import PyCRUD module
from CRUD_Python_Module import PyCRUD

###########################
# Data Manipulation / Model
###########################
USERNAME = "aacuser"
PASSWORD = "Dogs&Cat5"
DATABASE_NAME = "aac"
COLLECTION_NAME = "animals"

shelter = PyCRUD(USERNAME, PASSWORD)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read(DATABASE_NAME, COLLECTION_NAME, {}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display': 'none'}),
    html.Div([
        html.Div([
            html.A(
                html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'height': '120px'}),
                href='https://www.snhu.edu',
                target='_blank'
            )
        ], style={'width': '140px', 'display': 'flex', 'justifyContent': 'flex-start'}),
        html.Div([
            html.H1('SNHU CS-340 Dashboard', style={'margin': '0'}),
            html.H4('Coded by Danylo Herasymov', style={'margin': '8px 0 0 0'})
        ], style={'flex': '1', 'textAlign': 'center'}),
        html.Div(style={'width': '140px'})
    ], style={'display': 'flex', 'alignItems': 'center'}),
    html.Hr(),
    html.Div([
        html.Label('Interactive Rescue Filters'),
        dcc.Dropdown(id='filter-type', options=[
            {'label': 'Water Rescue', 'value': 'water'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
            {'label': 'Disaster Rescue or Individual Tracking', 'value': 'disaster'},
            {'label': 'Reset', 'value': 'reset'}
        ], value='reset', clearable=False)
    ], style={'width': '450px', 'margin': '0 0 20px 0'}),
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        filter_action='native',
        sort_action='native',
        sort_mode='multi',
        page_action='native',
        page_current=0,
        page_size=10,
        row_selectable='single',
        selected_rows=[0] if not df.empty else [],
        style_table={'overflowX': 'auto'},
        style_cell={
            'minWidth': '120px',
            'width': '120px',
            'maxWidth': '180px',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis',
            'textAlign': 'left'
        },
        style_header={'fontWeight': 'bold'}
    ),
    html.Br(),
    html.Hr(),
    html.Div([
        html.Div(id='graph-id', className='col s12 m6', style={'width': '50%'}),
        html.Div(id='map-id', className='col s12 m6', style={'width': '50%'})
    ], style={'display': 'flex', 'gap': '20px'})
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback([Output('datatable-id', 'data'),
               Output('datatable-id', 'columns')],
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    query = build_rescue_query(filter_type)
    update_df = pd.DataFrame.from_records(shelter.read(DATABASE_NAME, COLLECTION_NAME, query))

    if '_id' in update_df.columns:
        update_df.drop(columns=['_id'], inplace=True)

    columns = [{"name": i, "id": i, "deletable": False, "selectable": True} for i in update_df.columns]
    data = update_df.to_dict('records')

    return (data, columns)


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    gdf = pd.DataFrame.from_dict(viewData) if viewData is not None else df.copy()

    if gdf.empty or 'breed' not in gdf.columns:
        return [html.Div('No data available for the chart.')]

    breed_counts = gdf['breed'].fillna('Unknown').value_counts()

    if len(breed_counts) > 10:
        top_breeds = breed_counts.iloc[:10]
        other_total = breed_counts.iloc[10:].sum()
        breed_counts = pd.concat([top_breeds, pd.Series({'Other Breeds': other_total})])

    chart_df = breed_counts.reset_index()
    chart_df.columns = ['breed', 'count']
    figure = px.pie(chart_df, names='breed', values='count', title='Breed Distribution')
    figure.update_layout(height=550, legend=dict(orientation='v', y=0.0, x=1.0))

    return [
        dcc.Graph(
            figure=figure
        )
    ]


#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    if viewData is None:
        return

    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty:
        return [html.Div('No data available for the map.')]

    # Because we only allow single row selection, default to the first visible row.
    if not index:
        row = 0
    else:
        row = min(index[0], len(dff) - 1)

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row, 13], dff.iloc[row, 14]], children=[
                dl.Tooltip(dff.iloc[row, 4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row, 9])
                ])
            ])
        ])
    ]


def build_rescue_query(filter_type):
    rescue_queries = {
        'water': {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
            '$or': [
                {'breed': {'$regex': 'Labrador Retriever Mix', '$options': 'i'}},
                {'breed': {'$regex': 'Chesapeake Bay Retriever', '$options': 'i'}},
                {'breed': {'$regex': 'Newfoundland', '$options': 'i'}},
            ],
        },
        'mountain': {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
            '$or': [
                {'breed': {'$regex': 'German Shepherd', '$options': 'i'}},
                {'breed': {'$regex': 'Alaskan Malamute', '$options': 'i'}},
                {'breed': {'$regex': 'Old English Sheepdog', '$options': 'i'}},
                {'breed': {'$regex': 'Siberian Husky', '$options': 'i'}},
                {'breed': {'$regex': 'Rottweiler', '$options': 'i'}},
            ],
        },
        'disaster': {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300},
            '$or': [
                {'breed': {'$regex': 'Doberman Pinscher', '$options': 'i'}},
                {'breed': {'$regex': 'German Shepherd', '$options': 'i'}},
                {'breed': {'$regex': 'Golden Retriever', '$options': 'i'}},
                {'breed': {'$regex': 'Bloodhound', '$options': 'i'}},
                {'breed': {'$regex': 'Rottweiler', '$options': 'i'}},
            ],
        },
    }

    if filter_type == 'reset':
        return {}

    return rescue_queries.get(filter_type, {})


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server()

Dash app running on https://sisterlogo-radarfragile-3000.codio.io/proxy/8050/
